## LoadBalancer

In this notebook, we look at Services of type LoadBalancer in Kubernetes.

- Make sure you have a Kubernetes cluster (Docker Desktop) running.
- Also make sure you have installed the `kubectl` tool on your computer.

## List Nodes

- Note that Docker Desktop's Kubernetes cluster only has one node.

In [1]:
!kubectl get nodes -o wide

NAME             STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE         KERNEL-VERSION                       CONTAINER-RUNTIME
docker-desktop   Ready    control-plane   25d   v1.30.5   192.168.65.3   <none>        Docker Desktop   5.15.167.4-microsoft-standard-WSL2   docker://27.4.0


## Deploy a Deployment

- The Deployment definition is in the YAML file `manifests/deploy-app.yaml`.

In [2]:
!kubectl apply -f manifests/deploy-app.yaml

deployment.apps/deploy-nginx created


## Let's look at the Deployment's YAML

**Note:**

- The number of `replicas` is set to 3.
- The `revisionHistoryLimit` is set to 3.
- The Pod template's container is listening on `containerPort` 80.
- The Pod template's `labels` are `app: nginx` and `env: prod`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.
  - The Service's `selector` matches these labels
    - Therefore, the Service will load balance between the three Pod replicas.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx         # the deployment is named deploy-nginx
spec:
  replicas: 3                 # the deployment defines three replicas
  revisionHistoryLimit: 3     # the deployment defines a maximum of three ReplicaSet revisions to store in history
  selector:
    matchLabels:
      app: nginx              # the deployment has two matchLabels that match the Pod template's labels below
      env: prod               # app = nginx and env = prod
  template:
    metadata:
      name: myapp-pod
      labels:
        app: nginx             # the Pod template has two labels defined (the Service's selector matches these two labels):
        env: prod              # app = nginx and env = prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine    # the container is based on the nginx:alpine image
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi    
        ports:
        - containerPort: 80    # the Pod template's container is listening on containerPort 80
```

In [3]:
!type manifests\deploy-app.yaml
#!cat manifests/deploy-app.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx
  labels:
    app: nginx
    env: prod
spec:
  replicas: 3
  revisionHistoryLimit: 3
  selector:
    matchLabels:
      app: nginx
      env: prod
  template:
    metadata:
      labels:
        app: nginx
        env: prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine
        ports:
        - containerPort: 80
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi


## Deploy a Service

- The Service definition is in the YAML file `manifests/loadbalancer.yaml`.

In [4]:
!kubectl apply -f manifests/loadbalancer.yaml

service/svc-example created


## Let's look at the Service's YAML

**Note:**

- The service's name is `svc-example`.
- The service's is:
  - Of type `LoadBalancer` which means it will be assigned an external (public) IP address.
  - Listening on `port` 8080.
  - Redirecting traffic to `targetPort` 80.
- The servie's `selector` defines two labels (that match the Pod template's labels for the Deployment):
  - `app: nginx` and `env: prod`
  - This means the Service will (round robin) load balance between the three Pods in the Deployment's ReplicaSet.

```bash
apiVersion: v1
kind: Service
metadata:
  name: svc-example    # the Service's name is svc-example
spec:
  type: LoadBalancer   # the Service's type is LoadBalancer
  selector:
    app: nginx         # the Service's selector defines two labels (the Service's selector matches the Pod template's two labels):
    env: prod          # app = nginx and env = prod
  ports:
  - protocol: TCP      # the Service's protocol is TCP
    port: 8080         # the Service is listening for internal (private) traffic on port 8080
    targetPort: 80     # the Service is redirecting traffic to targetPort 80
```

In [5]:
!type manifests\loadbalancer.yaml
!cat manifests/loadbalancer.yaml # use this on Linux/Mac

apiVersion: v1
kind: Service
metadata:
  name: svc-example
spec:
  type: LoadBalancer
  selector:
    app: nginx
    env: prod
  ports:
  - protocol: TCP
    port: 8080
    targetPort: 80


'cat' is not recognized as an internal or external command,
operable program or batch file.


## Get Pods

- We see that 3 Pods are running.

In [6]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                            READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
deploy-nginx-6c867f559b-c78dw   1/1     Running   0          57s   10.1.1.25   docker-desktop   <none>           <none>
deploy-nginx-6c867f559b-kqxpq   1/1     Running   0          56s   10.1.1.27   docker-desktop   <none>           <none>
deploy-nginx-6c867f559b-rvfr2   1/1     Running   0          56s   10.1.1.26   docker-desktop   <none>           <none>


## Get Services

- Notice the `TYPE` is `LoadBalancer` for the `svc-example` Service.
- Notice the `EXTERNAL-IP` and `PORT` 8080 used by the `Loadbalancer`.
  - In Docker Desktop's Kubernetes cluster, `EXTERNAL-IP` will be `localhost`.
  - In a cloud provider's Kubernetes cluster, `EXTERNAL-IP` would be a proper external (public) IP Address.

In [7]:
#!kubectl get svc -o wide
!kubectl get services -o wide

NAME          TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE   SELECTOR
kubernetes    ClusterIP      10.96.0.1       <none>        443/TCP          25d   <none>
svc-example   LoadBalancer   10.106.62.201   localhost     8080:30117/TCP   42s   app=nginx,env=prod


## Access the Load Balancer Service

- Use the Load Balancer Service's assigned `EXTERNAL-IP` address above and the Service's port number 8080: `http://EXTERNAL-IP:8080`
- When using a cloud provider, you would use the cloud provider's LoadBalancer external IP address and your Service's port number.
  - In Docker Desktop's Kubernetes cluster, `EXTERNAL-IP` is `localhost`, so you would use `http://localhost:8080`

In [8]:
!curl http://localhost:8080

<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width: 35em; margin: 0 auto;
font-family: Tahoma, Verdana, Arial, sans-serif; }
</style>
</head>
<body>
<h1>Welcome to nginx!</h1>
<p>If you see this page, the nginx web server is successfully installed and
working. Further configuration is required.</p>

<p>For online documentation and support please refer to
<a href="http://nginx.org/">nginx.org</a>.<br/>
Commercial support is available at
<a href="http://nginx.com/">nginx.com</a>.</p>

<p><em>Thank you for using nginx.</em></p>
</body>
</html>


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   615  100   615    0     0   141k      0 --:--:-- --:--:-- --:--:--  150k


## Delete the Service and the Deployment

In [9]:
!kubectl delete -f manifests/loadbalancer.yaml
!kubectl delete -f manifests/deploy-app.yaml

service "svc-example" deleted
deployment.apps "deploy-nginx" deleted


## List Services, Deployments and Pods

- We see that the Service and Deployment with associated Pods were deleted.

In [10]:
#!kubectl get svc
#!kubectl get deploy
#!kubectl get po
!kubectl get services
!kubectl get deployments
!kubectl get pods

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   25d


No resources found in default namespace.
No resources found in default namespace.
